# Azure ML Offline Inference and RAG Evaluation

Compare the out-of-the-box Hugging Face model with the registered Azure ML fine-tuned model on the same held-out RAFT records. Generation runs locally; Azure AI Evaluation uses one Azure OpenAI judge configuration for **relevancy, groundedness, and coherence**.

> Run this notebook on a GPU host. Judge calls can incur Azure OpenAI cost. Start with a small `MAX_SAMPLES`, inspect row-level failures, and only then scale the run.

## 1. Install and Import Dependencies

Install the control-plane and evaluation extras into the active kernel once, then restart the kernel if packages were added. No credentials are written to the notebook.

In [ ]:
# Uncomment only when the active kernel is missing dependencies.
# %pip install -r ../requirements-azureml.txt

import gc
import json
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from azure.ai.evaluation import CoherenceEvaluator, GroundednessEvaluator, RelevanceEvaluator
from azure.identity import DefaultAzureCredential
from IPython.display import Markdown, display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "lib").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "lib").exists():
    raise RuntimeError("Start this notebook from the repository or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from lib.config import AzureMLConfig
from lib.prompts import SYSTEM_PROMPT, user_prompt

## 2. Configure Evaluation Parameters

Use an immutable registered-model version and a sealed test split. Both models receive identical prompts and deterministic generation settings. Thresholds are reporting flags, not production approval criteria.

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
REGISTERED_MODEL_NAME = "raft-llama32-1b"
REGISTERED_MODEL_VERSION = "1"
TEST_DATA_PATH = PROJECT_ROOT / "data" / "training_data_raft" / "test.jsonl"
DOWNLOAD_ROOT = PROJECT_ROOT / "output" / "downloaded_models"
REPORT_ROOT = PROJECT_ROOT / "output" / "offline_evaluation"

MAX_SAMPLES = 20
MAX_NEW_TOKENS = 384
MAX_INPUT_TOKENS = 4096
SEED = 42
JUDGE_DELAY_SECONDS = 0.0
QUALITY_THRESHOLD = 3.0
REGRESSION_TOLERANCE = 0.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else (
    torch.float16 if DEVICE == "cuda" else torch.float32
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "dtype": str(DTYPE), "max_samples": MAX_SAMPLES})

## 3. Authenticate with Azure Machine Learning

`DefaultAzureCredential` uses the Azure CLI identity locally and managed identity on Azure compute. The same credential is passed to Azure AI Evaluation; environment variables contain resource identifiers only.

In [ ]:
config = AzureMLConfig.from_env()
credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
ml_client = config.create_ml_client()

required_judge_env = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_GPT_DEPLOYMENT",
]
missing_judge_env = [name for name in required_judge_env if not os.getenv(name)]
if missing_judge_env:
    raise ValueError(f"Missing Azure OpenAI evaluation settings: {', '.join(missing_judge_env)}")

judge_model_config = {
    "type": "azure_openai",
    "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
    "azure_deployment": os.environ["AZURE_OPENAI_GPT_DEPLOYMENT"],
    "api_version": os.environ["AZURE_OPENAI_API_VERSION"],
}
workspace = ml_client.workspaces.get(config.workspace_name)
print("Workspace:", workspace.name)
print("Judge deployment:", judge_model_config["azure_deployment"])

## 4. Load and Validate the RAG Evaluation Dataset

The model input remains `instruction`, exactly as used during training. The evaluator receives the user question and a plain-text rendering of retrieved `context`; nested RAFT context structures are flattened without changing record order.

In [ ]:
REQUIRED_FIELDS = {"id", "type", "question", "context", "cot_answer", "instruction"}


def flatten_context(value):
    if isinstance(value, str):
        return [value.strip()] if value.strip() else []
    if isinstance(value, dict):
        parts = []
        for nested_value in value.values():
            parts.extend(flatten_context(nested_value))
        return parts
    if isinstance(value, (list, tuple)):
        parts = []
        for nested_value in value:
            parts.extend(flatten_context(nested_value))
        return parts
    return []


records = []
with TEST_DATA_PATH.open(encoding="utf-8") as stream:
    for line_number, line in enumerate(stream, start=1):
        if not line.strip():
            continue
        record = json.loads(line)
        missing = REQUIRED_FIELDS.difference(record)
        if missing:
            raise ValueError(f"Line {line_number} is missing fields: {sorted(missing)}")
        if not all(str(record[field]).strip() for field in ("question", "cot_answer", "instruction")):
            raise ValueError(f"Line {line_number} contains an empty required value")
        record["evaluation_context"] = "\n\n".join(flatten_context(record["context"]))
        if not record["evaluation_context"]:
            raise ValueError(f"Line {line_number} has no evaluable context")
        records.append(record)
        if len(records) >= MAX_SAMPLES:
            break

if not records:
    raise ValueError(f"No evaluation records found in {TEST_DATA_PATH}")
print(f"Loaded {len(records)} held-out records from {TEST_DATA_PATH}")
pd.DataFrame(records)[["id", "type", "question"]].head()

## 5. Load the OOB Hugging Face Model

The loader is reused for both candidates. CUDA uses automatic device placement and reduced precision; CPU uses float32. Access to gated Hugging Face models must already be configured outside this notebook.

In [ ]:
def load_local_model(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    tokenizer.padding_side = "right"
    model_kwargs = {"torch_dtype": DTYPE, "low_cpu_mem_usage": True}
    if DEVICE == "cuda":
        model_kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(model_path, **model_kwargs)
    if DEVICE == "cpu":
        model.to(DEVICE)
    model.eval()
    return model, tokenizer


def release_model(model, tokenizer):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


base_model, base_tokenizer = load_local_model(BASE_MODEL)
print("Loaded base model:", BASE_MODEL)

## 6. Run OOB Model Inference

Decode only newly generated tokens. Every source record produces an output row, including failures, so denominator changes cannot make a model look better.

In [ ]:
def generate_predictions(model, tokenizer, source_records, model_label):
    rows = []
    input_device = next(model.parameters()).device
    tokenizer_limit = getattr(tokenizer, "model_max_length", MAX_INPUT_TOKENS)
    input_limit = min(tokenizer_limit, MAX_INPUT_TOKENS)

    progress = tqdm(enumerate(source_records), total=len(source_records), desc=f"Generating: {model_label}")
    for sample_index, record in progress:
        row = {
            "sample_index": sample_index,
            "id": str(record["id"]),
            "type": str(record["type"]),
            "model": model_label,
            "query": str(record["question"]),
            "context": record["evaluation_context"],
            "ground_truth": str(record["cot_answer"]),
            "response": "",
            "latency_ms": np.nan,
            "inference_error": "",
        }
        try:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt(record["instruction"])},
            ]
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=input_limit,
            ).to(input_device)
            started = time.perf_counter()
            with torch.inference_mode():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            row["latency_ms"] = (time.perf_counter() - started) * 1000
            generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
            row["response"] = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        except Exception as exc:
            row["inference_error"] = f"{type(exc).__name__}: {exc}"
        rows.append(row)
    return pd.DataFrame(rows)


base_predictions = generate_predictions(base_model, base_tokenizer, records, "base")
base_predictions.to_csv(REPORT_ROOT / "base_predictions.csv", index=False)
display(base_predictions[["sample_index", "id", "type", "response", "latency_ms", "inference_error"]].head())
release_model(base_model, base_tokenizer)

## 7. Compute OOB RAG Evaluation Metrics

Azure AI Evaluation judge metrics use a 1-5 scale where higher is better. Relevancy measures how directly the response addresses the query, groundedness measures support from retrieved context, and coherence measures readability and organization. These are model-based estimates and must be complemented by human review.

In [ ]:
relevance_evaluator = RelevanceEvaluator(judge_model_config, credential=credential)
groundedness_evaluator = GroundednessEvaluator(judge_model_config, credential=credential)
coherence_evaluator = CoherenceEvaluator(judge_model_config, credential=credential)


def extract_score(result, metric):
    candidate_keys = (
        metric,
        f"{metric}_score",
        f"gpt_{metric}",
        f"gpt_{metric}_score",
    )
    for key in candidate_keys:
        value = result.get(key)
        if isinstance(value, (int, float, np.number)):
            return float(value)
    for key, value in result.items():
        if metric in key.lower() and isinstance(value, (int, float, np.number)):
            return float(value)
    raise KeyError(f"No numeric {metric} score in evaluator result keys: {sorted(result)}")


def evaluate_predictions(predictions):
    evaluated_rows = []
    for row in tqdm(predictions.to_dict("records"), desc="RAG evaluation"):
        scored = dict(row)
        errors = []
        for metric, evaluator, arguments in (
            ("relevancy", relevance_evaluator, {
                "query": row["query"], "response": row["response"], "context": row["context"]
            }),
            ("groundedness", groundedness_evaluator, {
                "query": row["query"], "response": row["response"], "context": row["context"]
            }),
            ("coherence", coherence_evaluator, {
                "query": row["query"], "response": row["response"]
            }),
        ):
            scored[metric] = np.nan
            scored[f"{metric}_reason"] = ""
            if row["inference_error"] or not row["response"]:
                errors.append(f"{metric}: skipped after inference failure")
                continue
            try:
                result = evaluator(**arguments)
                score_name = "relevance" if metric == "relevancy" else metric
                scored[metric] = extract_score(result, score_name)
                reason_key = next(
                    (key for key in result if "reason" in key.lower()), None
                )
                if reason_key:
                    scored[f"{metric}_reason"] = str(result[reason_key])
            except Exception as exc:
                errors.append(f"{metric}: {type(exc).__name__}: {exc}")
            if JUDGE_DELAY_SECONDS:
                time.sleep(JUDGE_DELAY_SECONDS)
        scored["metric_error"] = " | ".join(errors)
        evaluated_rows.append(scored)
    return pd.DataFrame(evaluated_rows)


base_evaluated = evaluate_predictions(base_predictions)
base_evaluated.to_csv(REPORT_ROOT / "base_evaluated.csv", index=False)
display(base_evaluated[["id", "relevancy", "groundedness", "coherence", "metric_error"]].head())

## 8. Download the Fine-Tuned Model from Azure ML

Resolve the immutable registry version, download its artifacts, and locate the merged Transformers directory. The checks reject incomplete assets before model allocation.

In [ ]:
registered_model = ml_client.models.get(
    name=REGISTERED_MODEL_NAME,
    version=REGISTERED_MODEL_VERSION,
)
print("Resolved model:", registered_model.id)

downloaded_path = Path(
    ml_client.models.download(
        name=REGISTERED_MODEL_NAME,
        version=REGISTERED_MODEL_VERSION,
        download_path=str(DOWNLOAD_ROOT),
    )
)
config_candidates = list(downloaded_path.rglob("config.json"))
model_directories = [
    path.parent
    for path in config_candidates
    if (path.parent / "tokenizer_config.json").exists()
    and (
        any(path.parent.glob("*.safetensors"))
        or any(path.parent.glob("pytorch_model*.bin"))
    )
]
if len(model_directories) != 1:
    raise RuntimeError(
        f"Expected one complete Transformers model under {downloaded_path}, "
        f"found {len(model_directories)}: {model_directories}"
    )
finetuned_model_path = model_directories[0]
print("Fine-tuned model directory:", finetuned_model_path)

## 9. Load the Fine-Tuned Model in Memory

Load the merged Azure ML artifact with the same device and precision policy used for the base model.

In [ ]:
finetuned_model, finetuned_tokenizer = load_local_model(finetuned_model_path)
print(
    "Loaded fine-tuned model:",
    f"{REGISTERED_MODEL_NAME}:{REGISTERED_MODEL_VERSION}",
)

## 10. Run Fine-Tuned Model Inference

Generate against the same ordered records. The shared function preserves identical prompt construction, truncation, decoding, and latency measurement.

In [ ]:
finetuned_predictions = generate_predictions(
    finetuned_model,
    finetuned_tokenizer,
    records,
    "fine_tuned",
)
finetuned_predictions.to_csv(REPORT_ROOT / "fine_tuned_predictions.csv", index=False)
display(
    finetuned_predictions[
        ["id", "type", "response", "latency_ms", "inference_error"]
    ].head()
)
release_model(finetuned_model, finetuned_tokenizer)

## 11. Compute Fine-Tuned RAG Evaluation Metrics

Apply the already-instantiated judges without changing deployment, prompts, metric definitions, or throttling.

In [ ]:
finetuned_evaluated = evaluate_predictions(finetuned_predictions)
finetuned_evaluated.to_csv(REPORT_ROOT / "fine_tuned_evaluated.csv", index=False)
display(
    finetuned_evaluated[
        ["sample_index", "id", "relevancy", "groundedness", "coherence", "metric_error"]
    ].head()
)

## 12. Build the Model Comparison Report

Aggregate scores show overall movement; paired deltas show what changed on each exact question. A quality win has a positive fine-tuned-minus-base delta. A latency win has a negative delta. Regression flags use `REGRESSION_TOLERANCE`.

In [ ]:
QUALITY_METRICS = ["relevancy", "groundedness", "coherence"]
all_evaluated = pd.concat([base_evaluated, finetuned_evaluated], ignore_index=True)

summary = (
    all_evaluated.groupby("model", sort=False)
    .agg(
        sample_count=("sample_index", "size"),
        evaluated_count=("relevancy", "count"),
        relevancy=("relevancy", "mean"),
        groundedness=("groundedness", "mean"),
        coherence=("coherence", "mean"),
        mean_latency_ms=("latency_ms", "mean"),
        inference_failures=("inference_error", lambda values: values.astype(bool).sum()),
        metric_failures=("metric_error", lambda values: values.astype(bool).sum()),
    )
    .reset_index()
)

PAIR_KEYS = ["sample_index", "id", "type", "query"]
paired = base_evaluated.merge(
    finetuned_evaluated,
    on=PAIR_KEYS,
    how="outer",
    validate="one_to_one",
    suffixes=("_base", "_fine_tuned"),
    indicator=True,
)
if not paired["_merge"].eq("both").all():
    raise RuntimeError("Base and fine-tuned outputs do not contain identical evaluation rows")

comparison_rows = []
for metric in QUALITY_METRICS + ["latency_ms"]:
    base_column = f"{metric}_base"
    fine_column = f"{metric}_fine_tuned"
    delta_column = f"{metric}_delta"
    paired[delta_column] = paired[fine_column] - paired[base_column]
    valid = paired[[base_column, fine_column]].notna().all(axis=1)
    delta = paired.loc[valid, delta_column]
    lower_is_better = metric == "latency_ms"
    wins = delta.lt(0) if lower_is_better else delta.gt(0)
    regressions = (
        delta.gt(REGRESSION_TOLERANCE)
        if lower_is_better
        else delta.lt(-REGRESSION_TOLERANCE)
    )
    comparison_rows.append(
        {
            "metric": metric,
            "base_mean": paired.loc[valid, base_column].mean(),
            "fine_tuned_mean": paired.loc[valid, fine_column].mean(),
            "mean_delta": delta.mean(),
            "paired_count": int(valid.sum()),
            "fine_tuned_win_rate": wins.mean() if len(wins) else np.nan,
            "regression_count": int(regressions.sum()),
        }
    )
comparison = pd.DataFrame(comparison_rows)
comparison["below_quality_threshold"] = comparison.apply(
    lambda row: bool(row["fine_tuned_mean"] < QUALITY_THRESHOLD)
    if row["metric"] in QUALITY_METRICS else False,
    axis=1,
)

display(summary.style.format(precision=3))
display(comparison.style.format(precision=3))

In [ ]:
quality_plot = summary.set_index("model")[QUALITY_METRICS].T
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [2, 1]})
quality_plot.plot(kind="bar", ax=axes[0], color=["#5b6573", "#147d64"])
axes[0].axhline(QUALITY_THRESHOLD, color="#b54708", linestyle="--", label="threshold")
axes[0].set(title="RAG judge metrics", ylabel="Mean score (1-5)", xlabel="")
axes[0].set_ylim(0, 5)
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend()
summary.set_index("model")[["mean_latency_ms"]].plot(
    kind="bar", ax=axes[1], color="#3b6ea8", legend=False
)
axes[1].set(title="Generation latency", ylabel="Mean milliseconds", xlabel="")
axes[1].tick_params(axis="x", rotation=0)
fig.tight_layout()
comparison_chart_path = REPORT_ROOT / "model_comparison.png"
fig.savefig(comparison_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 13. Save Evaluation Artifacts

Export row-level evidence, paired deltas, aggregate summaries, the chart, and a portable Markdown report. Set `UPLOAD_TO_AZUREML = True` only when the run should be recorded as a new MLflow evaluation run.

Interpret deltas alongside failure counts and row-level reasons. Judge metrics are nondeterministic estimates even at stable settings; this report is comparison evidence, not a production gate by itself.

In [ ]:
detailed_csv_path = REPORT_ROOT / "detailed_evaluation.csv"
detailed_json_path = REPORT_ROOT / "detailed_evaluation.json"
paired_csv_path = REPORT_ROOT / "paired_comparison.csv"
summary_csv_path = REPORT_ROOT / "model_summary.csv"
comparison_csv_path = REPORT_ROOT / "metric_comparison.csv"
report_path = REPORT_ROOT / "comparison_report.md"

all_evaluated.to_csv(detailed_csv_path, index=False)
all_evaluated.to_json(detailed_json_path, orient="records", indent=2)
paired.to_csv(paired_csv_path, index=False)
summary.to_csv(summary_csv_path, index=False)
comparison.to_csv(comparison_csv_path, index=False)

report = f"""# Offline RAG Model Comparison

## Configuration

- Base model: `{BASE_MODEL}`
- Fine-tuned model: `{REGISTERED_MODEL_NAME}:{REGISTERED_MODEL_VERSION}`
- Samples: {len(records)}
- Judge deployment: `{judge_model_config['azure_deployment']}`
- Quality threshold: {QUALITY_THRESHOLD:.2f}
- Generation device: `{DEVICE}`

## Aggregate Model Results

```csv
{summary.to_csv(index=False).strip()}
```

## Paired Metric Comparison

Fine-tuned delta is `fine_tuned - base`; negative latency delta is an improvement.

```csv
{comparison.to_csv(index=False).strip()}
```

## Failure Counts

```csv
{summary[['model', 'inference_failures', 'metric_failures']].to_csv(index=False).strip()}
```

## Artifacts

- `{detailed_csv_path.name}`: row-level responses, scores, reasons, and errors
- `{paired_csv_path.name}`: paired responses and per-question deltas
- `{summary_csv_path.name}`: aggregate model results
- `{comparison_csv_path.name}`: metric deltas, win rates, and regressions
- `{comparison_chart_path.name}`: quality and latency chart

## Interpretation

Azure AI Evaluation judge metrics use a 1-5 scale where higher is better. Review row-level evidence and judge reasons, especially failures and regressions. Model-based metrics can vary and do not replace human, safety, fairness, load, and cost evaluation.
"""
report_path.write_text(report, encoding="utf-8")

UPLOAD_TO_AZUREML = False
if UPLOAD_TO_AZUREML:
    import mlflow

    mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)
    with mlflow.start_run(run_name="offline-rag-model-comparison"):
        mlflow.log_params(
            {
                "base_model": BASE_MODEL,
                "registered_model": REGISTERED_MODEL_NAME,
                "registered_model_version": REGISTERED_MODEL_VERSION,
                "sample_count": len(records),
                "judge_deployment": judge_model_config["azure_deployment"],
            }
        )
        for result in comparison.to_dict("records"):
            metric_name = result["metric"]
            mlflow.log_metric(f"fine_tuned_{metric_name}", result["fine_tuned_mean"])
            mlflow.log_metric(f"delta_{metric_name}", result["mean_delta"])
        mlflow.log_artifacts(str(REPORT_ROOT), artifact_path="offline_evaluation")

print("Saved evaluation artifacts to:", REPORT_ROOT)
display(Markdown(report))

In [ ]:
import inspect
from azure.ai.evaluation import CoherenceEvaluator, GroundednessEvaluator, RelevanceEvaluator
print(inspect.signature(RelevanceEvaluator.__call__))
print(inspect.signature(GroundednessEvaluator.__call__))
print(inspect.signature(CoherenceEvaluator.__call__))

(model_config, *, credential=None, threshold=3, **kwargs)
(model_config, *, threshold=3, credential=None, **kwargs)
(model_config, *, threshold=3, credential=None, **kwargs)
